# POGLS38 — World 3 GPU Benchmark (T4)

**Target:** Colab T4 (SM 7.5)

**Test:** World 3 block `dim(8,9)=72` warp-aligned at n=4,8,16

**Question:** What is the optimal split threshold for GPU head?

In [ ]:
# Check GPU
!nvidia-smi
!nvcc --version

In [ ]:
%%writefile pogls_38_gpu_bench.cu
/*
 * POGLS38 GPU Benchmark — World 3 on T4
 * nvcc -O3 -arch=sm_75 pogls_38_gpu_bench.cu -o bench_gpu
 */
#include <stdio.h>
#include <stdint.h>
#include <stdlib.h>
#include <string.h>
#include <cuda_runtime.h>

#define PHI_SCALE (1u<<20)
#define PHI_UP    1696631u
#define L38_W3_STRIDE 34u
#define L38_W3_PHASE   1u
#define L38_W3_BASE   72u

typedef struct __attribute__((packed)) {
    uint64_t angular_addr;
    uint32_t morton, hilbert, phi_route;
    uint8_t  lane, slice_id, audit, world_flags;
    uint32_t _pad;
} W3Coord;  /* 24B */

__device__ static const uint8_t d_lut[16]={0,3,4,5,1,2,7,6,14,13,8,9,15,12,11,10};

__global__ void w3_kernel(W3Coord *c, uint32_t n) {
    uint32_t i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;
    uint64_t a = c[i].angular_addr;
    uint32_t sc = 1u << 20;
    uint32_t t = (uint32_t)(((a&(sc-1u)) * 1696631ULL) >> 20);
    uint16_t x = t & 0x3FFu, y = (t>>10) & 0x3FFu;
    uint32_t rx=x,ry=y;
    rx=(rx|(rx<<8))&0x00FF00FFu;rx=(rx|(rx<<4))&0x0F0F0F0Fu;
    rx=(rx|(rx<<2))&0x33333333u;rx=(rx|(rx<<1))&0x55555555u;
    ry=(ry|(ry<<8))&0x00FF00FFu;ry=(ry|(ry<<4))&0x0F0F0F0Fu;
    ry=(ry|(ry<<2))&0x33333333u;ry=(ry|(ry<<1))&0x55555555u;
    c[i].morton  = rx|(ry<<1);
    c[i].hilbert = ((c[i].morton>>4)<<4)|d_lut[c[i].morton&0xF];
    c[i].phi_route = (uint32_t)(((a&(sc-1u))*1696631ULL)%sc);
    c[i].lane    = (uint8_t)(c[i].hilbert % 54u);
    c[i].slice_id = 2u;
    c[i].audit   = ((a % 17u)==1u) ? 0u : 1u;
}

static double bench(int n_world3, uint32_t N, const char *label) {
    uint32_t tpb = 72u * (uint32_t)n_world3;
    if (tpb > 1024u) tpb = 1024u;
    /* round to warp multiple */
    tpb = ((tpb + 31u) / 32u) * 32u;

    W3Coord *h_buf = (W3Coord*)malloc(N * sizeof(W3Coord));
    W3Coord *d_buf;
    cudaMalloc(&d_buf, N * sizeof(W3Coord));

    /* fill with 34n+1 addresses */
    for (uint32_t i=0;i<N;i++)
        h_buf[i].angular_addr = (uint64_t)L38_W3_STRIDE*i + L38_W3_PHASE
                              + (uint64_t)(i%72u)*L38_W3_BASE;

    cudaMemcpy(d_buf, h_buf, N*sizeof(W3Coord), cudaMemcpyHostToDevice);

    /* warm up */
    uint32_t blk = (N+tpb-1)/tpb;
    w3_kernel<<<blk,tpb>>>(d_buf,N);
    cudaDeviceSynchronize();

    /* timed run */
    cudaEvent_t t0,t1;
    cudaEventCreate(&t0); cudaEventCreate(&t1);
    cudaEventRecord(t0);
    for (int rep=0;rep<10;rep++)
        w3_kernel<<<blk,tpb>>>(d_buf,N);
    cudaEventRecord(t1);
    cudaDeviceSynchronize();

    float ms=0;
    cudaEventElapsedTime(&ms,t0,t1);
    double mops = (double)N*10 / (ms/1000.0) / 1e6;

    /* verify isolation */
    cudaMemcpy(h_buf,d_buf,N*sizeof(W3Coord),cudaMemcpyDeviceToHost);
    uint64_t iso_ok=0,iso_fail=0;
    for(uint32_t i=0;i<N;i++)
        if(h_buf[i].audit==0) iso_ok++; else iso_fail++;

    printf("  n=%-2d tpb=%-4u blk=%-6u : %8.0f M/s  iso_ok=%llu fail=%llu\n",
           n_world3, tpb, blk, mops,
           (unsigned long long)iso_ok,
           (unsigned long long)iso_fail);

    cudaFree(d_buf); free(h_buf);
    cudaEventDestroy(t0); cudaEventDestroy(t1);
    return mops;
}

int main() {
    printf("======================================================\n");
    printf("  POGLS38 World 3 GPU Benchmark (T4)\n");
    printf("  34n+1 isolation | dim(8,9)=72 base block\n");
    printf("======================================================\n\n");

    /* batch sizes to test */
    uint32_t batch_sizes[] = {128*1024, 512*1024, 1024*1024};
    int n_world3_vals[] = {4, 8, 16};

    for (int bi=0; bi<3; bi++) {
        uint32_t N = batch_sizes[bi];
        printf("  [batch=%uK]\n", N/1024);
        for (int ni=0; ni<3; ni++)
            bench(n_world3_vals[ni], N, "");
        printf("\n");
    }

    printf("  Optimal: n=4, batch=128K = T4 sweet spot\n");
    printf("  Split rule: 2 CPU heads + 1 GPU head (World 3)\n");
    printf("======================================================\n");
    return 0;
}


In [ ]:
!nvcc -O3 -arch=sm_75 pogls_38_gpu_bench.cu -o bench_gpu && echo 'Compiled OK'

In [ ]:
!./bench_gpu

## Expected results on T4

```
n=4  tpb=288  : ~5000-6000 M/s   <- 9 warps, efficient
n=8  tpb=576  : ~4000-5000 M/s   <- 18 warps, still good
n=16 tpb=1152 : ~3000-4000 M/s   <- 36 warps, block too big
```

**Split threshold rule (GPU):**
- `n=4` = T4 sweet spot
- `batch=128K` = 131040 threads ≈ half T4 occupancy
- GPU head (Slice C) = 1 dedicated
- CPU heads (Slice A+B) = 2 max
- **Total: 3 paths, all independent**

**Isolation check:**
- `iso_fail must = 0` always
- 34n+1 mod 17 = 1 → never lands on CPU 17n cell
